In [ ]:
pip install anthropic litellm


In [ ]:
pip install litellm[proxy]

In [34]:
import time

In [19]:
#testing the connection to local proxy with a simple example

from anthropic import Anthropic

try:
    # 1. Point to the standard local port (No /anthropic suffix needed anymore!)
    client = Anthropic(
        base_url="http://localhost:8000",
        api_key="fake-key-for-sdk-validation"
    )

    # 2. Ask for the model name you defined in your config.yaml
    print("Sending request to local proxy...")
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022", 
        max_tokens=500,
        temperature=0.0,
        messages=[
            {"role": "user", "content": "who is the f1 driver for ferrari?"}
        ]
    )

    print("\n=== Response ===")
    print(response.content[0].text)
    print("================\n")

except Exception as e:
    print(f"\n❌ Error: {e}")

Sending request to local proxy...

=== Response ===
Ferrari currently has two F1 drivers:

*   **Charles Leclerc**
*   **Carlos Sainz**



In [26]:
#testing the connection to local proxy with a simple example

from anthropic import Anthropic

try:
    # 1. Point to the standard local port (No /anthropic suffix needed anymore!)
    client = Anthropic(
        base_url="http://localhost:8000",
        api_key="fake-key-for-sdk-validation"
    )
   
    messages=[]
    # 2. Ask for the model name you defined in your config.yaml
    def chat(messages,text):
        print("Sending request to local proxy...")
        response = client.messages.create(
        model="claude-3-5-sonnet-20241022", 
        max_tokens=500,
        temperature=0.0,
        messages=messages
    )
        
    def add_user_message(messages,text):
        user_message={"role": "user","content":text}
        messages.append(user_message)
    add_user_message(messages,"who is the f1 driver for ferrari?")
    
    

    print("\n=== Response ===")
    print(response.content[0].text)
    print("================\n")

except Exception as e:
    print(f"\n❌ Error: {e}")


=== Response ===
Ferrari currently has two F1 drivers:

*   **Charles Leclerc**
*   **Carlos Sainz**



In [31]:
import traceback
from anthropic import Anthropic

client = Anthropic(
    base_url="http://localhost:8000",
    api_key="fake-key-for-sdk-validation"
)

def add_user_message(messages,text):
    user_message={"role": "user","content":text}
    messages.append(user_message)

def add_assistant_message(messages,text):
    assistant_message={"role": "assistant","content":text}
    messages.append(assistant_message)

def chat(messages):
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022", # Your code will call this model name
        max_tokens=500,
        temperature=0.0,
        messages=messages
    )
    return response.content[0].text

messages=[]
add_user_message(messages,"What is the capital of India?")
print(messages)
try:
    answer = chat(messages)
    print(f"\n✅ Answer: {answer}")
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\n--- Full Stack Trace ---")
    traceback.print_exc()

[{'role': 'user', 'content': 'What is the capital of India?'}]

✅ Answer: The capital of India is **New Delhi**.


In [33]:
messages=[]
system = """ you are a patient tutor.
do not directly answer a student's questions.
Guide them to the solution step by step
"""

def chat(messages,system=None,temperature=0.5):
    param = {
        "model":"claude-3-5-sonnet-20241022",
        "max_tokens":500,
        "temperature": temperature,
        "messages":messages
    }
    if system:
        param["system"]=system
    response = client.messages.create(**param)
    return response.content[0].text


add_user_message(messages,"how to solve 5x+2=3")
answer = chat(messages,system=system,temperature=0.5)
print(answer)

That's a great question! It looks like you're trying to solve for 'x' in this equation.

Do you remember what our main goal is when we're trying to solve for a variable like 'x'? What do we want the equation to look like in the end?


TO HAVE STREAMING RESPONSE INTEAD OF WAITING FOR THE WHOLE REPONSE: The different type of Stream events are below ;


1.   Message Start
2.   ContentBlockStart
3.   contenctBlockDelta
4.   ContenctBlockStop
5.   MessageDelta
6.   MessageStop

In [37]:
messages=[]
add_user_message(messages,"write a 1 sentence description of a fake db")
responseStream = client.messages.create(
        model="claude-3-5-sonnet-20241022", # Your code will call this model name
        max_tokens=1000,
        messages=messages,
        stream=True
)
 # wait for the stream to start
for event in responseStream:
  print(event)

RawMessageStartEvent(message=Message(id='msg_77571a33-6944-48a1-9977-3c431c929487', container=None, content=[], model='gemini-2.5-flash', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo=None, input_tokens=0, output_tokens=0, server_tool_use=None, service_tier=None)), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='**The ChronoWeave is a sentient database that catalogs all possible future timelines, but perpetually corrupts its own data whenever observed, making every prediction immediately invalid.**', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockStopEvent(index=0, type='content_block_stop')
RawMessageDeltaEvent(delta=Delta(container=None, stop_details=None, stop_reaso

the above code will fetch all the stream provided by the api but we only need the RawContectBlockDeltaEvent

In [38]:
model="claude-3-5-sonnet-20241022"

In [40]:
# This will print the response chuck by chunk and only the response is printed as stream.
messages=[]
add_user_message(messages,"write a 1 sentence description of a fake db")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
#      print(text,end="")
#we can print the stream one by one:
        pass

stream.get_final_message()



ParsedMessage(id='msg_cb4b08d6-85d9-4260-97f9-61d125ba135b', container=None, content=[ParsedTextBlock(citations=None, text='', type='text', parsed_output=None)], model='gemini-2.5-flash', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo=None, input_tokens=0, output_tokens=0, server_tool_use=None, service_tier=None))

now to generate a code where we can copy it directly from AI,using copy snippet. we need to start with ```json and ends with ``` and get all the values in between them.


In [42]:
messages=[]
def chat(messages,system=None,temperature=0.5,stop_sequences=[]):
    param = {
        "model":"claude-3-5-sonnet-20241022",
        "max_tokens":500,
        "temperature": temperature,
        "messages":messages,
        "stop_sequences": stop_sequences
    }
    if system:
        param["system"]=system
    response = client.messages.create(**param)
    return response.content[0].text

add_user_message(messages,"Generate a very short event bridge rule as json")
add_assistant_message(messages,"```json")
text = chat(messages,stop_sequences=["```"])

In [43]:
print(text)


{
  "source": ["aws.s3"],
  "detail-type": ["Object Created"],
  "detail": {
    "bucket": {
      "name": ["my-cool-bucket"]
    }
  }
}



In [45]:
import json
#clean up and parse the json
clean_json = json.loads(text.strip())
print(clean_json)

{'source': ['aws.s3'], 'detail-type': ['Object Created'], 'detail': {'bucket': {'name': ['my-cool-bucket']}}}
